# E91 Entangled-Pair Measurement Demo

This notebook simulates the quantum-measurement part of an E91-style protocol using the repo's agent/runtime model.

A central source node emits entangled photon pairs. One photon travels to Alice and the other travels to Bob over separate quantum channels. Alice and Bob measure their received photons with randomly selected bases, and their local agents collect the detector reports.

We stop at measurement reports. This notebook does not implement sifting, Bell-test evaluation, error correction, privacy amplification, or final-key extraction.

In [1]:
from dataclasses import dataclass, field

import numpy as np

from simyuj.components import (
    ACTION_TRANSMIT_QUANTUM,
    Port,
    PortDelivery,
    QuantumChannel,
)
from simyuj.components.detectors import (
    ACTION_DETECT_SIGNAL,
    DetectionReport,
    DetectorArray,
    Measure,
    SinglePhotonDetector,
    SinglePhotonDetectorParams,
)
from simyuj.components.ports import PortDirection, PortKind
from simyuj.components.sources import EntangledPairSource, SourcePreparationReport
from simyuj.control.payloads import AgentStart
from simyuj.control import AGENT_REPORT, NodeAgent, SessionRuntime, AgentContext
from simyuj.engine import Timeline
from simyuj.network import Network, Node
from simyuj.qstate import StateSampler
from simyuj.qstate.noise import depolarizing
from simyuj.qstate.measure import MeasurementBasis
from simyuj.primitives.units import seconds_to_ticks

In [2]:
@dataclass(frozen=True)
class E91DetectionRecord:
    node_id: str
    time: int
    signal_id: str
    measurement_label: str | None
    outcome: str | None
    success: bool

In [3]:
@dataclass(slots=True)
class E91SourceAgent(NodeAgent):
    source: EntangledPairSource
    preparations: list[SourcePreparationReport] = field(default_factory=list)

    def on_start(self, start: AgentStart, ctx: AgentContext) -> None:
        del start
        self.source.schedule_start(ctx.timeline)

    def on_report(self, report: object, ctx: AgentContext) -> None:
        del ctx
        if not isinstance(report, SourcePreparationReport):
            raise TypeError(f"unsupported source report: {type(report)!r}")
        self.preparations.append(report)

In [4]:
@dataclass(slots=True)
class E91ReceiverAgent(NodeAgent):
    detections: list[E91DetectionRecord] = field(default_factory=list)
    reports: list[DetectionReport] = field(default_factory=list)

    def on_report(self, report: object, ctx: AgentContext) -> None:
        del ctx
        if not isinstance(report, DetectionReport):
            raise TypeError(f"unsupported detector report: {type(report)!r}")
        
        self.reports.append(report)
        self.detections.append(
            E91DetectionRecord(
                node_id=self.node_id,
                time=report.time,
                signal_id=report.signal_id,
                measurement_label=report.measurement_label,
                outcome=report.outcome,
                success=report.success,
            )
        )

In [5]:
def linear_polarization_basis(name: str, angle_deg: float) -> MeasurementBasis:
    """Linear polarization basis rotated by angle_deg from the Z basis."""
    theta = np.deg2rad(angle_deg)

    ket_0 = np.array([np.cos(theta), np.sin(theta)], dtype=np.complex128)
    ket_1 = np.array([-np.sin(theta), np.cos(theta)], dtype=np.complex128)

    return MeasurementBasis(
        name=name,
        vectors=(ket_0, ket_1),
        labels=("0", "1"),
    )


scenario = {
    "name": "metro_fiber_10km_each_side",
    "master_seed": 2026,

    # Source model. This gives 5,000 emission attempts at 50 MHz.
    # With emission_probability < 1, only some attempts create pairs.
    "num_slots": 50_000,
    "source_frequency_hz": 50e6,
    "emission_probability": 0.9,
    "wavelength_nm": 1550.0,

    # Central source with Alice and Bob 10 km away.
    "alice_distance_m": 1_000.0,
    "bob_distance_m": 1_000.0,

    # Typical telecom fiber scale.
    "fiber_speed_m_per_s": 2.0e8,
    "attenuation_db_per_km": 0.2,
    "fixed_insertion_loss_db": 1.0,
    "channel_timing_jitter_stddev_s": 50e-12,

    # Single-qubit depolarizing noise applied on each fiber arm.
    # Use 0.0 for a clean channel; increase it to model disturbance/Eve.
    "channel_depolarizing_probability": 0.03,

    # Simple single-photon detector model.
    "detector_efficiency": 0.90,
    "dark_count_rate_hz": 100.0,
    "detector_dead_time_s": 50e-9,
    "detector_jitter_stddev_s": 80e-12,
    "detection_window_s": 1e-9,
}

scenario["duration_s"] = scenario["num_slots"] / scenario["source_frequency_hz"]

alice_basis_specs = (
    ("A0_0", 0.0),
    ("A1_pi4", np.pi / 4),
    ("A2_pi2", np.pi / 2),
)

bob_basis_specs = (
    ("B0_pi4", np.pi / 4),
    ("B1_pi2", np.pi / 2),
    ("B2_3pi4", 3 * np.pi / 4),
)

def photon_basis_from_e91_angle(name: str, e91_angle_rad: float) -> MeasurementBasis:
    physical_polarizer_angle_rad = e91_angle_rad / 2
    return linear_polarization_basis(
        name,
        np.rad2deg(physical_polarizer_angle_rad),
    )

alice_bases = tuple(
    (
        photon_basis_from_e91_angle(name, angle_rad),
        1 / len(alice_basis_specs),
    )
    for name, angle_rad in alice_basis_specs
)

bob_bases = tuple(
    (
        photon_basis_from_e91_angle(name, angle_rad),
        1 / len(bob_basis_specs),
    )
    for name, angle_rad in bob_basis_specs
)

basis_angles = {
    **{
        basis.name: angle_rad
        for (basis, _prob), (_name, angle_rad)
        in zip(alice_bases, alice_basis_specs)
    },
    **{
        basis.name: angle_rad
        for (basis, _prob), (_name, angle_rad)
        in zip(bob_bases, bob_basis_specs)
    },
}

alice_channel_transmission = 10 ** (
    -(
        scenario["attenuation_db_per_km"] * scenario["alice_distance_m"] / 1000
        + scenario["fixed_insertion_loss_db"]
    )
    / 10
)
bob_channel_transmission = 10 ** (
    -(
        scenario["attenuation_db_per_km"] * scenario["bob_distance_m"] / 1000
        + scenario["fixed_insertion_loss_db"]
    )
    / 10
)

print("Scenario:", scenario["name"])
print("Duration (s):", scenario["duration_s"])
print("Expected emitted pairs:", scenario["num_slots"] * scenario["emission_probability"])
print("Channel depolarizing probability:", scenario["channel_depolarizing_probability"])
print("Alice channel transmission:", alice_channel_transmission)
print("Bob channel transmission:", bob_channel_transmission)
print(
    "Expected both photons detected per emitted pair:",
    alice_channel_transmission
    * bob_channel_transmission
    * scenario["detector_efficiency"] ** 2,
)

Scenario: metro_fiber_10km_each_side
Duration (s): 0.001
Expected emitted pairs: 45000.0
Channel depolarizing probability: 0.03
Alice channel transmission: 0.7585775750291838
Bob channel transmission: 0.7585775750291838
Expected both photons detected per emitted pair: 0.46610634924309713


In [6]:
def e91_measurement_choices(bases: tuple[tuple[MeasurementBasis, float], ...]):
    return tuple(
        (Measure.basis(basis, label=basis.name), probability)
        for basis, probability in bases
    )


def e91_readout_for_bases(
    bases: tuple[tuple[MeasurementBasis, float], ...],
    detector_zero_id: str,
    detector_one_id: str,
) -> dict[str, dict[str, str]]:
    return {
        basis.name: {
            "0": detector_zero_id,
            "1": detector_one_id,
        }
        for basis, _probability in bases
    }


def make_receiver_detector(
    *,
    node_name: str,
    bases: tuple[tuple[MeasurementBasis, float], ...],
) -> DetectorArray:
    detector_zero_id = f"{node_name}_D0"
    detector_one_id = f"{node_name}_D1"

    detector_params = SinglePhotonDetectorParams(
        efficiency=scenario["detector_efficiency"],
        dark_count_rate_hz=scenario["dark_count_rate_hz"],
        dead_time_ticks=seconds_to_ticks(scenario["detector_dead_time_s"]),
        jitter_stddev_ticks=seconds_to_ticks(scenario["detector_jitter_stddev_s"]),
        photon_number_resolving=False,
    )

    return DetectorArray(
        device_id=f"{node_name}_detector",
        detectors=(
            SinglePhotonDetector(detector_zero_id, params=detector_params),
            SinglePhotonDetector(detector_one_id, params=detector_params),
        ),
        measurement=Measure.random(
            e91_measurement_choices(bases),
            label=f"{node_name}_basis_choice",
        ),
        readout=e91_readout_for_bases(
            bases,
            detector_zero_id=detector_zero_id,
            detector_one_id=detector_one_id,
        ),
        detection_window_ticks=seconds_to_ticks(scenario["detection_window_s"]),
        consume_signal=True,
    )


pair_sampler = StateSampler(
    states=("psi-",),
    probabilities=(1.0,),
    rep="ket",
    labels=("psi-",),
)

source = EntangledPairSource(
    device_id="e91_source",
    frequency_hz=scenario["source_frequency_hz"],
    emission_probability=scenario["emission_probability"],
    wavelength_nm=scenario["wavelength_nm"],
    duration_s=scenario["duration_s"],
    sampler=pair_sampler,
)

alice_channel = QuantumChannel(
    channel_id="source_to_alice_fiber",
    length_m=scenario["alice_distance_m"],
    propagation_speed_m_per_s=scenario["fiber_speed_m_per_s"],
    attenuation_db_per_km=scenario["attenuation_db_per_km"],
    fixed_insertion_loss_db=scenario["fixed_insertion_loss_db"],
    timing_jitter_stddev_ticks=seconds_to_ticks(
        scenario["channel_timing_jitter_stddev_s"]
    ),
    noise_models=(
        depolarizing(scenario["channel_depolarizing_probability"]),
    ),
)

bob_channel = QuantumChannel(
    channel_id="source_to_bob_fiber",
    length_m=scenario["bob_distance_m"],
    propagation_speed_m_per_s=scenario["fiber_speed_m_per_s"],
    attenuation_db_per_km=scenario["attenuation_db_per_km"],
    fixed_insertion_loss_db=scenario["fixed_insertion_loss_db"],
    timing_jitter_stddev_ticks=seconds_to_ticks(
        scenario["channel_timing_jitter_stddev_s"]
    ),
    noise_models=(
        depolarizing(scenario["channel_depolarizing_probability"]),
    ),
)

alice_detector = make_receiver_detector(node_name="alice", bases=alice_bases)
bob_detector = make_receiver_detector(node_name="bob", bases=bob_bases)

print("Source emission period ticks:", source.emission_period_ticks)
print("Alice channel delay ticks:", alice_channel.resolved_delay_ticks)
print("Bob channel delay ticks:", bob_channel.resolved_delay_ticks)
print("Alice detector:", alice_detector.device_id)
print("Bob detector:", bob_detector.device_id)

Source emission period ticks: 20000
Alice channel delay ticks: 5000000
Bob channel delay ticks: 5000000
Alice detector: alice_detector
Bob detector: bob_detector


In [7]:
source_agent = E91SourceAgent(
    agent_id="source_agent",
    node_id="source",
    source=source
)

alice_agent = E91ReceiverAgent(
    agent_id="alice_agent",
    node_id="alice"
)

bob_agent = E91ReceiverAgent(
    agent_id="bob_agent",
    node_id="bob"
)

network = Network("e91_three_note_network")

source_node = Node("source")
alice_node = Node("alice")
bob_node = Node("bob")

source_node.add_device("pair_source", source)
source_node.add_agent(source_agent)

alice_node.add_device("detector", alice_detector)
alice_node.add_agent(alice_agent)

bob_node.add_device("detector", bob_detector)
bob_node.add_agent(bob_agent)

network.add_node(source_node)
network.add_node(alice_node)
network.add_node(bob_node)

network.add_quantum_link(
    "source_to_alice_quantum_link",
    source_node_id="source",
    target_node_id="alice",
    channel=alice_channel
)

network.add_quantum_link(
    "source_to_bob_quantum_link",
    source_node_id="source",
    target_node_id="bob",
    channel=bob_channel
)

network.wire_ports(
    "source_left_to_alice_channel",
    source_port=source.left_output_port,
    target_port=alice_channel.input_port,
    target_action=ACTION_TRANSMIT_QUANTUM
)

network.wire_ports(
    "alice_channel_to_detector",
    source_port=alice_channel.output_port,
    target_port=alice_detector.input_port,
    target_action=ACTION_DETECT_SIGNAL
)


network.wire_ports(
    "source_right_to_bob_channel",
    source_port=source.right_output_port,
    target_port=bob_channel.input_port,
    target_action=ACTION_TRANSMIT_QUANTUM
)

network.wire_ports(
    "bob_channel_to_detector",
    source_port=bob_channel.output_port,
    target_port=bob_detector.input_port,
    target_action=ACTION_DETECT_SIGNAL
)


network.wire_ports(
    "source_report_to_source_agent",
    source.report_port,
    source_agent.report_port,
    target_action=AGENT_REPORT,
)

network.wire_ports(
    "alice_detector_report_to_agent",
    alice_detector.output_port,
    alice_agent.report_port,
    target_action=AGENT_REPORT,
)

network.wire_ports(
    "bob_detector_report_to_agent",
    bob_detector.output_port,
    bob_agent.report_port,
    target_action=AGENT_REPORT,
)

print("Nodes:", tuple(network.nodes))
print("Quantum links:", tuple(network.quantum_links))
print("Runtime wires:", tuple(network.wires))

Nodes: ('source', 'alice', 'bob')
Quantum links: ('source_to_alice_quantum_link', 'source_to_bob_quantum_link')
Runtime wires: ('source_left_to_alice_channel', 'alice_channel_to_detector', 'source_right_to_bob_channel', 'bob_channel_to_detector', 'source_report_to_source_agent', 'alice_detector_report_to_agent', 'bob_detector_report_to_agent')


In [8]:
timeline = Timeline(master_seed=scenario["master_seed"])

runtime = SessionRuntime(
    timeline=timeline,
    network=network,
    session_id=f"e91_{scenario['name']}"
)

runtime.run()

alice_successes = sum(record.success for record in alice_agent.detections)
bob_successes = sum(record.success for record in bob_agent.detections)

print("Simulation complete")
print("Timeline final time:", timeline.current_time)
print()
print("Prepared entangled pairs:", len(source_agent.preparations))
print()
print("Alice channel received:", alice_channel.received_count)
print("Alice channel delivered:", alice_channel.delivered_count)
print("Alice channel lost:", alice_channel.lost_count)
print("Alice detector reports:", len(alice_agent.reports))
print("Alice successful detections:", alice_successes)
print()
print("Bob channel received:", bob_channel.received_count)
print("Bob channel delivered:", bob_channel.delivered_count)
print("Bob channel lost:", bob_channel.lost_count)
print("Bob detector reports:", len(bob_agent.reports))
print("Bob successful detections:", bob_successes)

Simulation complete
Timeline final time: 1004961000

Prepared entangled pairs: 45034

Alice channel received: 45034
Alice channel delivered: 34096
Alice channel lost: 10938
Alice detector reports: 34096
Alice successful detections: 19026

Bob channel received: 45034
Bob channel delivered: 34275
Bob channel lost: 10759
Bob detector reports: 34275
Bob successful detections: 19141


In [9]:
def pair_index_from_signal_id(signal_id: str) -> int:
    # Example: "e91_source:pair:17:left"
    parts = signal_id.split(":")
    if len(parts) < 4 or parts[1] != "pair":
        raise ValueError(f"unexpected E91 signal id: {signal_id!r}")
    return int(parts[2])


alice_by_pair = {
    pair_index_from_signal_id(record.signal_id): record
    for record in alice_agent.detections
}

bob_by_pair = {
    pair_index_from_signal_id(record.signal_id): record
    for record in bob_agent.detections
}

paired_pair_ids = sorted(set(alice_by_pair) & set(bob_by_pair))

paired_measurements = [
    {
        "pair_id": pair_id,
        "alice_basis": alice_by_pair[pair_id].measurement_label,
        "alice_outcome": alice_by_pair[pair_id].outcome,
        "alice_success": alice_by_pair[pair_id].success,
        "bob_basis": bob_by_pair[pair_id].measurement_label,
        "bob_outcome": bob_by_pair[pair_id].outcome,
        "bob_success": bob_by_pair[pair_id].success,
    }
    for pair_id in paired_pair_ids
]

coincident_measurements = [
    row
    for row in paired_measurements
    if row["alice_success"] and row["bob_success"]
]

print("Pairs prepared:", len(source_agent.preparations))
print("Pairs with Alice report:", len(alice_by_pair))
print("Pairs with Bob report:", len(bob_by_pair))
print("Pairs with both reports:", len(paired_measurements))
print("Pairs with both successful detections:", len(coincident_measurements))

coincident_measurements[:10]

Pairs prepared: 45034
Pairs with Alice report: 34096
Pairs with Bob report: 34275
Pairs with both reports: 25915
Pairs with both successful detections: 8629


[{'pair_id': 1,
  'alice_basis': 'a0_0',
  'alice_outcome': '1',
  'alice_success': True,
  'bob_basis': 'b0_pi4',
  'bob_outcome': '0',
  'bob_success': True},
 {'pair_id': 3,
  'alice_basis': 'a0_0',
  'alice_outcome': '0',
  'alice_success': True,
  'bob_basis': 'b0_pi4',
  'bob_outcome': '1',
  'bob_success': True},
 {'pair_id': 7,
  'alice_basis': 'a1_pi4',
  'alice_outcome': '0',
  'alice_success': True,
  'bob_basis': 'b1_pi2',
  'bob_outcome': '1',
  'bob_success': True},
 {'pair_id': 20,
  'alice_basis': 'a2_pi2',
  'alice_outcome': '1',
  'alice_success': True,
  'bob_basis': 'b2_3pi4',
  'bob_outcome': '0',
  'bob_success': True},
 {'pair_id': 22,
  'alice_basis': 'a1_pi4',
  'alice_outcome': '0',
  'alice_success': True,
  'bob_basis': 'b1_pi2',
  'bob_outcome': '1',
  'bob_success': True},
 {'pair_id': 35,
  'alice_basis': 'a1_pi4',
  'alice_outcome': '0',
  'alice_success': True,
  'bob_basis': 'b2_3pi4',
  'bob_outcome': '1',
  'bob_success': True},
 {'pair_id': 46,
  'a

In [10]:
def same_orientation(alice_basis: str, bob_basis: str) -> bool:
    return np.isclose(basis_angles[alice_basis], basis_angles[bob_basis])


same_orientation_measurements = [
    row
    for row in coincident_measurements
    if same_orientation(row["alice_basis"], row["bob_basis"])
]

different_orientation_measurements = [
    row
    for row in coincident_measurements
    if not same_orientation(row["alice_basis"], row["bob_basis"])
]

print("Coincident detections:", len(coincident_measurements))
print("Same-orientation measurements for raw key:", len(same_orientation_measurements))
print("Different-orientation measurements for Bell test:", len(different_orientation_measurements))

same_orientation_measurements[:10]


Coincident detections: 8629
Same-orientation measurements for raw key: 2016
Different-orientation measurements for Bell test: 6613


[{'pair_id': 71,
  'alice_basis': 'a2_pi2',
  'alice_outcome': '1',
  'alice_success': True,
  'bob_basis': 'b1_pi2',
  'bob_outcome': '0',
  'bob_success': True},
 {'pair_id': 165,
  'alice_basis': 'a2_pi2',
  'alice_outcome': '0',
  'alice_success': True,
  'bob_basis': 'b1_pi2',
  'bob_outcome': '1',
  'bob_success': True},
 {'pair_id': 214,
  'alice_basis': 'a2_pi2',
  'alice_outcome': '1',
  'alice_success': True,
  'bob_basis': 'b1_pi2',
  'bob_outcome': '0',
  'bob_success': True},
 {'pair_id': 242,
  'alice_basis': 'a1_pi4',
  'alice_outcome': '1',
  'alice_success': True,
  'bob_basis': 'b0_pi4',
  'bob_outcome': '0',
  'bob_success': True},
 {'pair_id': 274,
  'alice_basis': 'a1_pi4',
  'alice_outcome': '0',
  'alice_success': True,
  'bob_basis': 'b0_pi4',
  'bob_outcome': '1',
  'bob_success': True},
 {'pair_id': 301,
  'alice_basis': 'a1_pi4',
  'alice_outcome': '1',
  'alice_success': True,
  'bob_basis': 'b0_pi4',
  'bob_outcome': '0',
  'bob_success': True},
 {'pair_id'

In [11]:
def bit_from_outcome(outcome: str) -> int:
    if outcome == "0":
        return 0
    if outcome == "1":
        return 1
    raise ValueError(f"unexpected outcome: {outcome!r}")


raw_key_rows = []

for row in same_orientation_measurements:
    alice_bit = bit_from_outcome(row["alice_outcome"])
    bob_bit = bit_from_outcome(row["bob_outcome"])

    # The psi- singlet is anticorrelated for equal analyzer orientations.
    bob_corrected_bit = 1 - bob_bit

    raw_key_rows.append(
        {
            "pair_id": row["pair_id"],
            "basis": row["alice_basis"],
            "alice_bit": alice_bit,
            "bob_raw_bit": bob_bit,
            "bob_corrected_bit": bob_corrected_bit,
            "bits_match_after_flip": alice_bit == bob_corrected_bit,
        }
    )

raw_key = [row["alice_bit"] for row in raw_key_rows]
raw_key_agreement = (
    sum(row["bits_match_after_flip"] for row in raw_key_rows) / len(raw_key_rows)
    if raw_key_rows
    else float("nan")
)
qber = 1 - raw_key_agreement

print("Raw key candidate length:", len(raw_key))
print("Alice/Bob agreement after Bob flips:", raw_key_agreement)
print("QBER:", qber)

raw_key_rows[:10]


Raw key candidate length: 2016
Alice/Bob agreement after Bob flips: 0.9756944444444444
QBER: 0.02430555555555558


[{'pair_id': 71,
  'basis': 'a2_pi2',
  'alice_bit': 1,
  'bob_raw_bit': 0,
  'bob_corrected_bit': 1,
  'bits_match_after_flip': True},
 {'pair_id': 165,
  'basis': 'a2_pi2',
  'alice_bit': 0,
  'bob_raw_bit': 1,
  'bob_corrected_bit': 0,
  'bits_match_after_flip': True},
 {'pair_id': 214,
  'basis': 'a2_pi2',
  'alice_bit': 1,
  'bob_raw_bit': 0,
  'bob_corrected_bit': 1,
  'bits_match_after_flip': True},
 {'pair_id': 242,
  'basis': 'a1_pi4',
  'alice_bit': 1,
  'bob_raw_bit': 0,
  'bob_corrected_bit': 1,
  'bits_match_after_flip': True},
 {'pair_id': 274,
  'basis': 'a1_pi4',
  'alice_bit': 0,
  'bob_raw_bit': 1,
  'bob_corrected_bit': 0,
  'bits_match_after_flip': True},
 {'pair_id': 301,
  'basis': 'a1_pi4',
  'alice_bit': 1,
  'bob_raw_bit': 0,
  'bob_corrected_bit': 1,
  'bits_match_after_flip': True},
 {'pair_id': 316,
  'basis': 'a1_pi4',
  'alice_bit': 0,
  'bob_raw_bit': 1,
  'bob_corrected_bit': 0,
  'bits_match_after_flip': True},
 {'pair_id': 365,
  'basis': 'a2_pi2',
  '

In [ ]:
def outcome_value(outcome: str) -> int:
    if outcome == "0":
        return 1
    if outcome == "1":
        return -1
    raise ValueError(f"unexpected binary outcome: {outcome!r}")


def observed_correlation(rows: list[dict]) -> float:
    if not rows:
        return float("nan")

    products = [
        outcome_value(row["alice_outcome"]) * outcome_value(row["bob_outcome"])
        for row in rows
    ]
    return sum(products) / len(products)


def ideal_psi_minus_correlation(alice_basis: str, bob_basis: str) -> float:
    return float(-np.cos(basis_angles[alice_basis] - basis_angles[bob_basis]))


bell_test_pairs = {
    "E_A0_B0": (alice_bases[0][0].name, bob_bases[0][0].name),
    "E_A0_B2": (alice_bases[0][0].name, bob_bases[2][0].name),
    "E_A2_B0": (alice_bases[2][0].name, bob_bases[0][0].name),
    "E_A2_B2": (alice_bases[2][0].name, bob_bases[2][0].name),
}

bell_rows = {}

for label, (alice_basis, bob_basis) in bell_test_pairs.items():
    rows = [
        row
        for row in different_orientation_measurements
        if row["alice_basis"] == alice_basis and row["bob_basis"] == bob_basis
    ]

    bell_rows[label] = {
        "alice_basis": alice_basis,
        "bob_basis": bob_basis,
        "coincidences": len(rows),
        "observed_E": observed_correlation(rows),
        "ideal_E": ideal_psi_minus_correlation(alice_basis, bob_basis),
    }

observed_s = (
    bell_rows["E_A0_B0"]["observed_E"]
    - bell_rows["E_A0_B2"]["observed_E"]
    + bell_rows["E_A2_B0"]["observed_E"]
    + bell_rows["E_A2_B2"]["observed_E"]
)

ideal_s = (
    bell_rows["E_A0_B0"]["ideal_E"]
    - bell_rows["E_A0_B2"]["ideal_E"]
    + bell_rows["E_A2_B0"]["ideal_E"]
    + bell_rows["E_A2_B2"]["ideal_E"]
)

{
    "bell_rows": bell_rows,
    "observed_S": observed_s,
    "abs_observed_S": abs(observed_s),
    "ideal_S": ideal_s,
    "abs_ideal_S": abs(ideal_s),
    "classical_bound": 2.0,
    "tsirelson_bound": 2 * np.sqrt(2),
}


{'bell_rows': {'E_A0_B0': {'alice_basis': 'a0_0',
   'bob_basis': 'b0_pi4',
   'coincidences': 964,
   'observed_E': -0.7095435684647303,
   'ideal_E': -0.7071067811865476},
  'E_A0_B2': {'alice_basis': 'a0_0',
   'bob_basis': 'b2_3pi4',
   'coincidences': 889,
   'observed_E': 0.5838020247469067,
   'ideal_E': 0.7071067811865475},
  'E_A2_B0': {'alice_basis': 'a2_pi2',
   'bob_basis': 'b0_pi4',
   'coincidences': 959,
   'observed_E': -0.7247132429614181,
   'ideal_E': -0.7071067811865476},
  'E_A2_B2': {'alice_basis': 'a2_pi2',
   'bob_basis': 'b2_3pi4',
   'coincidences': 982,
   'observed_E': -0.7331975560081466,
   'ideal_E': -0.7071067811865476}},
 'observed_S': -2.751256392181202,
 'abs_observed_S': 2.751256392181202,
 'ideal_S': -2.82842712474619,
 'abs_ideal_S': 2.82842712474619,
 'classical_bound': 2.0,
 'tsirelson_bound': np.float64(2.8284271247461903)}

Depolarizing channel noise is used here as a compact proxy for random channel disturbance or an Eve-like interaction. It reduces Bell-test contrast and introduces disagreements in the same-orientation raw-key candidates.


In [14]:
analysis_summary = {
    "scenario": scenario["name"],
    "source_state": source_agent.preparations[0].sampler_label if source_agent.preparations else None,
    "master_seed": scenario["master_seed"],
    "num_slots": scenario["num_slots"],
    "emission_probability": scenario["emission_probability"],
    "channel_depolarizing_probability": scenario["channel_depolarizing_probability"],
    "prepared_pairs": len(source_agent.preparations),
    "paired_reports": len(paired_measurements),
    "coincident_detections": len(coincident_measurements),
    "same_orientation_raw_key_candidates": len(same_orientation_measurements),
    "different_orientation_bell_test_samples": len(different_orientation_measurements),
    "raw_key_candidate_length": len(raw_key),
    "raw_key_agreement_after_bob_flip": raw_key_agreement,
    "qber": qber,
    "observed_S": observed_s,
    "abs_observed_S": abs(observed_s),
    "ideal_S": ideal_s,
    "abs_ideal_S": abs(ideal_s),
    "bell_visibility": abs(observed_s) / abs(ideal_s),
    "classical_bound": 2.0,
    "tsirelson_bound": 2 * np.sqrt(2),
    "bell_violation_observed": abs(observed_s) > 2.0,
}

analysis_summary


{'scenario': 'metro_fiber_10km_each_side',
 'source_state': 'psi-',
 'master_seed': 2026,
 'num_slots': 50000,
 'emission_probability': 0.9,
 'channel_depolarizing_probability': 0.03,
 'prepared_pairs': 45034,
 'paired_reports': 25915,
 'coincident_detections': 8629,
 'same_orientation_raw_key_candidates': 2016,
 'different_orientation_bell_test_samples': 6613,
 'raw_key_candidate_length': 2016,
 'raw_key_agreement_after_bob_flip': 0.9756944444444444,
 'qber': 0.02430555555555558,
 'observed_S': -2.751256392181202,
 'abs_observed_S': 2.751256392181202,
 'ideal_S': -2.82842712474619,
 'abs_ideal_S': 2.82842712474619,
 'bell_visibility': 0.9727160258470817,
 'classical_bound': 2.0,
 'tsirelson_bound': np.float64(2.8284271247461903),
 'bell_violation_observed': True}